<a href="https://colab.research.google.com/github/ChanhLDV/CIC_Dashboard/blob/main/Financial_Model_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Interactive 3-Statement Financial Model
### With Assumptions & Scenarios — Built for Google Colab

**Run all cells** (`Runtime → Run all`) and scroll down to interact with the model.

Drag sliders to adjust assumptions in real-time. Try the preset scenarios: *Base Case*, *Bull Case*, *Bear Case*, *Aggressive Growth*.

In [1]:
# Install dependencies (Colab usually has most of these pre-installed)
!pip install ipywidgets plotly pandas numpy -q

# Enable widgets in Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # Not in Colab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00


In [2]:
# ============================================================
# 📊 INTERACTIVE 3-STATEMENT FINANCIAL MODEL
# With Assumptions & Scenarios - Built for Google Colab
# ============================================================
# Just run this cell and scroll down to see the magic!
# ============================================================

# Install dependencies (uncomment in Colab if needed)
# !pip install ipywidgets pandas numpy matplotlib plotly -q

import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interactive_output, HBox, VBox, Layout, GridBox
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STYLING
# ============================================================
display(HTML("""
<style>
    .model-header {
        background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%);
        color: white;
        padding: 20px;
        border-radius: 10px;
        font-family: 'Segoe UI', sans-serif;
        margin-bottom: 20px;
        box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    }
    .section-header {
        background: #f1f5f9;
        color: #1e293b;
        padding: 12px 16px;
        border-left: 5px solid #3b82f6;
        font-family: 'Segoe UI', sans-serif;
        font-weight: 600;
        font-size: 16px;
        margin: 15px 0 10px 0;
        border-radius: 4px;
    }
    .widget-label { min-width: 200px !important; font-family: 'Segoe UI', sans-serif; }
    .widget-readout { font-weight: 600 !important; color: #1e40af !important; }
</style>
"""))

# ============================================================
# PRESETS (Scenarios)
# ============================================================
PRESETS = {
    'Base Case':       dict(rev_growth=4.0, cogs=45.0, salaries=15.0, rent=50000, marketing=5.0,
                            depreciation=20000, other_opex=3.0, interest=10000, tax=25.0,
                            dso=35, dio=25, dpo=30, capex=3.0, dividends=10.0),
    'Bull Case':       dict(rev_growth=8.0, cogs=40.0, salaries=13.0, rent=50000, marketing=6.0,
                            depreciation=20000, other_opex=2.5, interest=10000, tax=25.0,
                            dso=30, dio=20, dpo=35, capex=4.0, dividends=15.0),
    'Bear Case':       dict(rev_growth=1.0, cogs=52.0, salaries=18.0, rent=55000, marketing=4.0,
                            depreciation=20000, other_opex=4.0, interest=12000, tax=25.0,
                            dso=45, dio=35, dpo=25, capex=2.0, dividends=5.0),
    'Aggressive Growth': dict(rev_growth=12.0, cogs=48.0, salaries=20.0, rent=60000, marketing=10.0,
                            depreciation=25000, other_opex=3.5, interest=15000, tax=25.0,
                            dso=40, dio=30, dpo=30, capex=8.0, dividends=0.0),
}

# ============================================================
# FINANCIAL MODEL ENGINE
# ============================================================
def build_model(rev_growth, cogs, salaries, rent, marketing, depreciation, other_opex,
                interest, tax, dso, dio, dpo, capex, dividends, months=12):
    """Build the full 3-statement financial model."""

    # Starting values (tuned so opening BS balances: Assets = Liab + Equity)
    starting_revenue = 300_000
    starting_cash = 500_000
    starting_ppe = 800_000
    starting_equity = 1_100_000   # Plug so BS balances
    starting_retained = 200_000

    months_list = [f"M{i+1}" for i in range(months)]

    # ===== INCOME STATEMENT =====
    revenue = [starting_revenue * (1 + rev_growth/100) ** i for i in range(months)]
    cogs_val = [r * cogs/100 for r in revenue]
    gross_profit = [r - c for r, c in zip(revenue, cogs_val)]

    salaries_val = [r * salaries/100 for r in revenue]
    rent_val = [rent] * months
    marketing_val = [r * marketing/100 for r in revenue]
    depreciation_val = [depreciation] * months
    other_opex_val = [r * other_opex/100 for r in revenue]

    total_opex = [s + rt + m + d + o for s, rt, m, d, o in
                  zip(salaries_val, rent_val, marketing_val, depreciation_val, other_opex_val)]

    ebit = [g - o for g, o in zip(gross_profit, total_opex)]
    interest_val = [interest] * months
    ebt = [e - i for e, i in zip(ebit, interest_val)]
    tax_val = [max(0, e) * tax/100 for e in ebt]
    net_income = [e - t for e, t in zip(ebt, tax_val)]

    # Margins
    gross_margin = [g/r*100 if r > 0 else 0 for g, r in zip(gross_profit, revenue)]
    ebit_margin = [e/r*100 if r > 0 else 0 for e, r in zip(ebit, revenue)]
    net_margin = [n/r*100 if r > 0 else 0 for n, r in zip(net_income, revenue)]

    # ===== BALANCE SHEET =====
    ar = [r * dso/30 for r in revenue]  # Accounts Receivable
    inventory = [c * dio/30 for c in cogs_val]  # Inventory
    ap = [c * dpo/30 for c in cogs_val]  # Accounts Payable

    # PPE
    capex_val = [r * capex/100 for r in revenue]
    ppe = [starting_ppe]
    for i in range(months):
        ppe.append(ppe[-1] + capex_val[i] - depreciation_val[i])
    ppe = ppe[1:]

    # Retained Earnings
    dividends_val = [max(0, n) * dividends/100 for n in net_income]
    retained = [starting_retained]
    for i in range(months):
        retained.append(retained[-1] + net_income[i] - dividends_val[i])
    retained = retained[1:]

    # ===== CASH FLOW =====
    # Operating CF
    cfo = []
    prev_ar, prev_inv, prev_ap = 0, 0, 0
    for i in range(months):
        d_ar = ar[i] - prev_ar
        d_inv = inventory[i] - prev_inv
        d_ap = ap[i] - prev_ap
        cfo_i = net_income[i] + depreciation_val[i] - d_ar - d_inv + d_ap
        cfo.append(cfo_i)
        prev_ar, prev_inv, prev_ap = ar[i], inventory[i], ap[i]

    cfi = [-c for c in capex_val]
    cff = [-d for d in dividends_val]
    net_cf = [o + i + f for o, i, f in zip(cfo, cfi, cff)]

    cash = [starting_cash]
    for n in net_cf:
        cash.append(cash[-1] + n)
    cash = cash[1:]

    # Total Assets & Liab+Equity
    total_assets = [c + a + i + p for c, a, i, p in zip(cash, ar, inventory, ppe)]
    total_liab = ap
    total_equity = [starting_equity + r for r in retained]

    # Build DataFrames
    is_df = pd.DataFrame({
        'Revenue': revenue, 'COGS': cogs_val, 'Gross Profit': gross_profit,
        'Salaries': salaries_val, 'Rent': rent_val, 'Marketing': marketing_val,
        'Depreciation': depreciation_val, 'Other OpEx': other_opex_val,
        'Total OpEx': total_opex, 'EBIT': ebit, 'Interest': interest_val,
        'EBT': ebt, 'Tax': tax_val, 'Net Income': net_income,
        'Gross Margin %': gross_margin, 'EBIT Margin %': ebit_margin, 'Net Margin %': net_margin,
    }, index=months_list).T

    bs_df = pd.DataFrame({
        'Cash': cash, 'Accounts Receivable': ar, 'Inventory': inventory,
        'PP&E': ppe, 'Total Assets': total_assets,
        'Accounts Payable': ap, 'Total Liabilities': total_liab,
        'Equity': [starting_equity]*months, 'Retained Earnings': retained,
        'Total Equity': total_equity,
    }, index=months_list).T

    cf_df = pd.DataFrame({
        'Net Income': net_income, 'Depreciation': depreciation_val,
        'Operating CF': cfo, 'Investing CF': cfi, 'Financing CF': cff,
        'Net Change in Cash': net_cf, 'Ending Cash': cash,
    }, index=months_list).T

    return is_df, bs_df, cf_df, revenue, net_income, cash, ebit, ebit_margin, net_margin


def format_financial_df(df, highlight_rows=None, pct_rows=None):
    """Format DataFrame with accounting style."""
    highlight_rows = highlight_rows or []
    pct_rows = pct_rows or []

    def format_val(val, is_pct=False):
        if is_pct:
            return f"{val:.1f}%"
        if abs(val) < 1:
            return f"{val:,.3f}"
        return f"{val:,.0f}"

    styled = df.copy()
    for idx in styled.index:
        is_pct = idx in pct_rows
        styled.loc[idx] = [format_val(v, is_pct) for v in df.loc[idx]]

    def style_rows(row):
        name = row.name
        if name in highlight_rows:
            return ['background-color: #1e3a8a; color: white; font-weight: 700; padding: 8px;'] * len(row)
        if name in pct_rows:
            return ['color: #3b82f6; font-style: italic;'] * len(row)
        if name.startswith('Total') or name in ['Gross Profit', 'EBIT', 'EBT']:
            return ['background-color: #e0e7ff; font-weight: 600;'] * len(row)
        return [''] * len(row)

    return (styled.style
            .apply(style_rows, axis=1)
            .set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', 'white'),
                                              ('font-weight', '600'), ('padding', '10px'),
                                              ('font-family', 'Segoe UI')]},
                {'selector': 'td', 'props': [('padding', '8px 12px'), ('text-align', 'right'),
                                              ('font-family', 'Segoe UI'), ('border-bottom', '1px solid #e2e8f0')]},
                {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('background-color', '#f8fafc'),
                                                         ('color', '#1e293b'), ('font-weight', '500')]},
            ]))


# ============================================================
# WIDGETS
# ============================================================
slider_layout = Layout(width='450px')
style_slider = {'description_width': '200px'}

# Preset dropdown
preset = widgets.Dropdown(
    options=list(PRESETS.keys()) + ['Custom'],
    value='Base Case', description='📋 Load Preset:',
    style={'description_width': '120px'}, layout=Layout(width='350px')
)

view_mode = widgets.Dropdown(
    options=['All Statements', 'Income Statement', 'Balance Sheet', 'Cash Flow', 'Charts Only'],
    value='All Statements', description='👁 View:',
    style={'description_width': '120px'}, layout=Layout(width='350px')
)

# Income Statement sliders
rev_growth = widgets.FloatSlider(value=4.0, min=-5, max=20, step=0.5, description='Revenue Growth (% / month)',
                                  style=style_slider, layout=slider_layout, readout_format='.1f')
cogs = widgets.FloatSlider(value=45.0, min=20, max=80, step=1, description='COGS (% of Revenue)',
                            style=style_slider, layout=slider_layout, readout_format='.1f')
salaries = widgets.FloatSlider(value=15.0, min=5, max=40, step=1, description='Salaries (% of Revenue)',
                                style=style_slider, layout=slider_layout, readout_format='.1f')
rent = widgets.IntSlider(value=50000, min=10000, max=200000, step=5000, description='Rent ($/month)',
                          style=style_slider, layout=slider_layout)
marketing = widgets.FloatSlider(value=5.0, min=0, max=20, step=0.5, description='Marketing (% of Revenue)',
                                 style=style_slider, layout=slider_layout, readout_format='.1f')
depreciation = widgets.IntSlider(value=20000, min=0, max=100000, step=1000, description='Depreciation ($/month)',
                                  style=style_slider, layout=slider_layout)
other_opex = widgets.FloatSlider(value=3.0, min=0, max=15, step=0.5, description='Other OpEx (% of Revenue)',
                                  style=style_slider, layout=slider_layout, readout_format='.1f')
interest = widgets.IntSlider(value=10000, min=0, max=50000, step=1000, description='Interest Expense ($/month)',
                              style=style_slider, layout=slider_layout)
tax = widgets.FloatSlider(value=25.0, min=0, max=50, step=1, description='Tax Rate (%)',
                           style=style_slider, layout=slider_layout, readout_format='.1f')

# Balance Sheet & Cash Flow sliders
dso = widgets.IntSlider(value=35, min=0, max=120, step=1, description='Days Sales Outstanding',
                         style=style_slider, layout=slider_layout)
dio = widgets.IntSlider(value=25, min=0, max=120, step=1, description='Days Inventory Outstanding',
                         style=style_slider, layout=slider_layout)
dpo = widgets.IntSlider(value=30, min=0, max=120, step=1, description='Days Payable Outstanding',
                         style=style_slider, layout=slider_layout)
capex = widgets.FloatSlider(value=3.0, min=0, max=20, step=0.5, description='CapEx (% of Revenue)',
                             style=style_slider, layout=slider_layout, readout_format='.1f')
dividends = widgets.FloatSlider(value=10.0, min=0, max=100, step=5, description='Dividends (% of Net Income)',
                                 style=style_slider, layout=slider_layout, readout_format='.1f')

# Preset handler
def apply_preset(change):
    if change['new'] != 'Custom' and change['new'] in PRESETS:
        vals = PRESETS[change['new']]
        rev_growth.value = vals['rev_growth']; cogs.value = vals['cogs']
        salaries.value = vals['salaries']; rent.value = vals['rent']
        marketing.value = vals['marketing']; depreciation.value = vals['depreciation']
        other_opex.value = vals['other_opex']; interest.value = vals['interest']
        tax.value = vals['tax']; dso.value = vals['dso']; dio.value = vals['dio']
        dpo.value = vals['dpo']; capex.value = vals['capex']; dividends.value = vals['dividends']

preset.observe(apply_preset, names='value')

# ============================================================
# MAIN RENDER FUNCTION
# ============================================================
def render(rev_growth, cogs, salaries, rent, marketing, depreciation, other_opex,
           interest, tax, dso, dio, dpo, capex, dividends, view_mode):

    is_df, bs_df, cf_df, revenue, net_income, cash, ebit, ebit_margin, net_margin = build_model(
        rev_growth, cogs, salaries, rent, marketing, depreciation, other_opex,
        interest, tax, dso, dio, dpo, capex, dividends
    )

    # KPI Cards
    total_rev = sum(revenue)
    total_ni = sum(net_income)
    ending_cash = cash[-1]
    avg_margin = np.mean(net_margin)

    kpi_html = f"""
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin: 20px 0;">
        <div style="background: linear-gradient(135deg, #3b82f6, #1e40af); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
            <div style="font-size: 12px; opacity: 0.9; font-family: Segoe UI;">TOTAL REVENUE (12M)</div>
            <div style="font-size: 26px; font-weight: 700; margin-top: 8px; font-family: Segoe UI;">${total_rev:,.0f}</div>
        </div>
        <div style="background: linear-gradient(135deg, #10b981, #047857); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
            <div style="font-size: 12px; opacity: 0.9; font-family: Segoe UI;">NET INCOME (12M)</div>
            <div style="font-size: 26px; font-weight: 700; margin-top: 8px; font-family: Segoe UI;">${total_ni:,.0f}</div>
        </div>
        <div style="background: linear-gradient(135deg, #8b5cf6, #6d28d9); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
            <div style="font-size: 12px; opacity: 0.9; font-family: Segoe UI;">ENDING CASH</div>
            <div style="font-size: 26px; font-weight: 700; margin-top: 8px; font-family: Segoe UI;">${ending_cash:,.0f}</div>
        </div>
        <div style="background: linear-gradient(135deg, #f59e0b, #d97706); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
            <div style="font-size: 12px; opacity: 0.9; font-family: Segoe UI;">AVG NET MARGIN</div>
            <div style="font-size: 26px; font-weight: 700; margin-top: 8px; font-family: Segoe UI;">{avg_margin:.1f}%</div>
        </div>
    </div>
    """
    display(HTML(kpi_html))

    # Charts
    if view_mode in ['All Statements', 'Charts Only']:
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Revenue & Net Income', 'Margin Trends', 'Cash Balance', 'EBIT over Time'),
            specs=[[{'secondary_y': False}]*2, [{'secondary_y': False}]*2]
        )
        months_list = [f"M{i+1}" for i in range(12)]

        fig.add_trace(go.Bar(x=months_list, y=revenue, name='Revenue', marker_color='#3b82f6'), 1, 1)
        fig.add_trace(go.Scatter(x=months_list, y=net_income, name='Net Income', mode='lines+markers',
                                  line=dict(color='#10b981', width=3)), 1, 1)

        fig.add_trace(go.Scatter(x=months_list, y=ebit_margin, name='EBIT Margin %', mode='lines+markers',
                                  line=dict(color='#8b5cf6', width=3)), 1, 2)
        fig.add_trace(go.Scatter(x=months_list, y=net_margin, name='Net Margin %', mode='lines+markers',
                                  line=dict(color='#f59e0b', width=3)), 1, 2)

        fig.add_trace(go.Scatter(x=months_list, y=cash, name='Cash', mode='lines+markers',
                                  fill='tozeroy', line=dict(color='#06b6d4', width=3)), 2, 1)

        fig.add_trace(go.Bar(x=months_list, y=ebit, name='EBIT',
                             marker_color=['#10b981' if v >= 0 else '#ef4444' for v in ebit]), 2, 2)

        fig.update_layout(height=600, showlegend=True, template='plotly_white',
                           title_text="<b>Financial Performance Dashboard</b>",
                           title_font_size=18, font=dict(family='Segoe UI'))
        fig.show()

    # Tables
    if view_mode in ['All Statements', 'Income Statement']:
        display(HTML('<div class="section-header">📊 Income Statement</div>'))
        display(format_financial_df(
            is_df,
            highlight_rows=['Net Income'],
            pct_rows=['Gross Margin %', 'EBIT Margin %', 'Net Margin %']
        ))

    if view_mode in ['All Statements', 'Balance Sheet']:
        display(HTML('<div class="section-header">🏦 Balance Sheet</div>'))
        display(format_financial_df(bs_df, highlight_rows=['Total Assets', 'Total Equity']))

    if view_mode in ['All Statements', 'Cash Flow']:
        display(HTML('<div class="section-header">💰 Cash Flow Statement</div>'))
        display(format_financial_df(cf_df, highlight_rows=['Ending Cash']))


# ============================================================
# LAYOUT
# ============================================================
display(HTML("""
<div class="model-header">
    <div style="font-size: 28px; font-weight: 700;">📊 3-Statement Financial Model</div>
    <div style="font-size: 14px; opacity: 0.9; margin-top: 5px;">
        Interactive model with Assumptions, Scenarios, and Real-time Analysis
    </div>
</div>
"""))

display(HTML('<div class="section-header">⚙️ Scenario & View Controls</div>'))
display(HBox([preset, view_mode]))

display(HTML('<div class="section-header">📈 Income Statement Assumptions</div>'))
is_left = VBox([rev_growth, cogs, salaries, rent, marketing])
is_right = VBox([depreciation, other_opex, interest, tax])
display(HBox([is_left, is_right]))

display(HTML('<div class="section-header">🏦 Balance Sheet & Cash Flow Assumptions</div>'))
bs_left = VBox([dso, dio, dpo])
bs_right = VBox([capex, dividends])
display(HBox([bs_left, bs_right]))

# Live output
out = interactive_output(render, {
    'rev_growth': rev_growth, 'cogs': cogs, 'salaries': salaries, 'rent': rent,
    'marketing': marketing, 'depreciation': depreciation, 'other_opex': other_opex,
    'interest': interest, 'tax': tax, 'dso': dso, 'dio': dio, 'dpo': dpo,
    'capex': capex, 'dividends': dividends, 'view_mode': view_mode
})

display(out)


Output()

In [3]:
# ============================================================================
# 📊 INTERACTIVE 3-STATEMENT FINANCIAL MODEL — v2
# With Assumptions, Scenarios, AND Advanced Forecasting:
#   • Prophet  • LSTM Neural Network  • ARIMA  • Monte Carlo  • Ensemble
# Built for Google Colab
# ============================================================================

# Install (Colab): uncomment the line below on first run
# !pip install ipywidgets plotly prophet tensorflow statsmodels scikit-learn -q

import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import interactive_output, HBox, VBox, Layout, Tab
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Advanced forecasting imports — graceful fallback if unavailable
FORECAST_AVAILABLE = {'prophet': False, 'lstm': False, 'arima': False}
try:
    from prophet import Prophet
    FORECAST_AVAILABLE['prophet'] = True
except ImportError:
    pass

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    FORECAST_AVAILABLE['arima'] = True
except ImportError:
    pass

try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from sklearn.preprocessing import MinMaxScaler
    tf.get_logger().setLevel('ERROR')
    FORECAST_AVAILABLE['lstm'] = True
except ImportError:
    pass

# ============================================================================
# STYLING
# ============================================================================
display(HTML("""
<style>
    .model-header {
        background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 50%, #8b5cf6 100%);
        color: white; padding: 24px; border-radius: 12px;
        font-family: 'Segoe UI', sans-serif; margin-bottom: 20px;
        box-shadow: 0 6px 20px rgba(0,0,0,0.18);
    }
    .section-header {
        background: #f1f5f9; color: #1e293b; padding: 12px 16px;
        border-left: 5px solid #3b82f6; font-family: 'Segoe UI', sans-serif;
        font-weight: 600; font-size: 16px; margin: 15px 0 10px 0; border-radius: 4px;
    }
    .ml-header {
        background: #fef3c7; color: #78350f; padding: 12px 16px;
        border-left: 5px solid #f59e0b; font-family: 'Segoe UI', sans-serif;
        font-weight: 600; font-size: 16px; margin: 15px 0 10px 0; border-radius: 4px;
    }
    .widget-label { min-width: 200px !important; font-family: 'Segoe UI', sans-serif; }
    .widget-readout { font-weight: 600 !important; color: #1e40af !important; }
    .status-good { color: #059669; font-weight: 600; }
    .status-bad { color: #dc2626; font-weight: 600; }
</style>
"""))

# ============================================================================
# PRESETS
# ============================================================================
PRESETS = {
    'Base Case':         dict(rev_growth=4.0, cogs=45.0, salaries=15.0, rent=50000, marketing=5.0,
                              depreciation=20000, other_opex=3.0, interest=10000, tax=25.0,
                              dso=35, dio=25, dpo=30, capex=3.0, dividends=10.0),
    'Bull Case':         dict(rev_growth=8.0, cogs=40.0, salaries=13.0, rent=50000, marketing=6.0,
                              depreciation=20000, other_opex=2.5, interest=10000, tax=25.0,
                              dso=30, dio=20, dpo=35, capex=4.0, dividends=15.0),
    'Bear Case':         dict(rev_growth=1.0, cogs=52.0, salaries=18.0, rent=55000, marketing=4.0,
                              depreciation=20000, other_opex=4.0, interest=12000, tax=25.0,
                              dso=45, dio=35, dpo=25, capex=2.0, dividends=5.0),
    'Aggressive Growth': dict(rev_growth=12.0, cogs=48.0, salaries=20.0, rent=60000, marketing=10.0,
                              depreciation=25000, other_opex=3.5, interest=15000, tax=25.0,
                              dso=40, dio=30, dpo=30, capex=8.0, dividends=0.0),
}

# ============================================================================
# CORE FINANCIAL MODEL ENGINE
# ============================================================================
def build_model(rev_growth, cogs, salaries, rent, marketing, depreciation, other_opex,
                interest, tax, dso, dio, dpo, capex, dividends, months=12, seasonality=False):
    """Build the full 3-statement model. Optional seasonality adds realistic monthly variation."""
    starting_revenue = 300_000
    starting_cash = 500_000
    starting_ppe = 800_000
    starting_equity = 1_100_000
    starting_retained = 200_000

    months_list = [f"M{i+1}" for i in range(months)]

    # Seasonality multipliers (retail-ish pattern)
    if seasonality:
        season = np.array([0.92, 0.88, 0.95, 1.00, 1.03, 1.05, 1.02, 1.00, 1.04, 1.08, 1.12, 1.15])
        season = np.tile(season, (months // 12) + 1)[:months]
    else:
        season = np.ones(months)

    revenue = [starting_revenue * (1 + rev_growth/100) ** i * season[i] for i in range(months)]
    cogs_val = [r * cogs/100 for r in revenue]
    gross_profit = [r - c for r, c in zip(revenue, cogs_val)]
    salaries_val = [r * salaries/100 for r in revenue]
    rent_val = [rent] * months
    marketing_val = [r * marketing/100 for r in revenue]
    depreciation_val = [depreciation] * months
    other_opex_val = [r * other_opex/100 for r in revenue]
    total_opex = [s+rt+m+d+o for s,rt,m,d,o in zip(salaries_val, rent_val, marketing_val, depreciation_val, other_opex_val)]
    ebit = [g - o for g, o in zip(gross_profit, total_opex)]
    interest_val = [interest] * months
    ebt = [e - i for e, i in zip(ebit, interest_val)]
    tax_val = [max(0, e) * tax/100 for e in ebt]
    net_income = [e - t for e, t in zip(ebt, tax_val)]

    gross_margin = [g/r*100 if r > 0 else 0 for g, r in zip(gross_profit, revenue)]
    ebit_margin = [e/r*100 if r > 0 else 0 for e, r in zip(ebit, revenue)]
    net_margin = [n/r*100 if r > 0 else 0 for n, r in zip(net_income, revenue)]

    ar = [r * dso/30 for r in revenue]
    inventory = [c * dio/30 for c in cogs_val]
    ap = [c * dpo/30 for c in cogs_val]

    capex_val = [r * capex/100 for r in revenue]
    ppe = [starting_ppe]
    for i in range(months):
        ppe.append(ppe[-1] + capex_val[i] - depreciation_val[i])
    ppe = ppe[1:]

    dividends_val = [max(0, n) * dividends/100 for n in net_income]
    retained = [starting_retained]
    for i in range(months):
        retained.append(retained[-1] + net_income[i] - dividends_val[i])
    retained = retained[1:]

    cfo = []
    prev_ar, prev_inv, prev_ap = 0, 0, 0
    for i in range(months):
        d_ar = ar[i] - prev_ar; d_inv = inventory[i] - prev_inv; d_ap = ap[i] - prev_ap
        cfo.append(net_income[i] + depreciation_val[i] - d_ar - d_inv + d_ap)
        prev_ar, prev_inv, prev_ap = ar[i], inventory[i], ap[i]
    cfi = [-c for c in capex_val]
    cff = [-d for d in dividends_val]
    net_cf = [o+i+f for o,i,f in zip(cfo, cfi, cff)]
    cash = [starting_cash]
    for n in net_cf: cash.append(cash[-1] + n)
    cash = cash[1:]

    total_assets = [c+a+i+p for c,a,i,p in zip(cash, ar, inventory, ppe)]
    total_equity = [starting_equity + r for r in retained]

    is_df = pd.DataFrame({
        'Revenue': revenue, 'COGS': cogs_val, 'Gross Profit': gross_profit,
        'Salaries': salaries_val, 'Rent': rent_val, 'Marketing': marketing_val,
        'Depreciation': depreciation_val, 'Other OpEx': other_opex_val,
        'Total OpEx': total_opex, 'EBIT': ebit, 'Interest': interest_val,
        'EBT': ebt, 'Tax': tax_val, 'Net Income': net_income,
        'Gross Margin %': gross_margin, 'EBIT Margin %': ebit_margin, 'Net Margin %': net_margin,
    }, index=months_list).T

    bs_df = pd.DataFrame({
        'Cash': cash, 'Accounts Receivable': ar, 'Inventory': inventory, 'PP&E': ppe,
        'Total Assets': total_assets, 'Accounts Payable': ap, 'Total Liabilities': ap,
        'Equity': [starting_equity]*months, 'Retained Earnings': retained, 'Total Equity': total_equity,
    }, index=months_list).T

    cf_df = pd.DataFrame({
        'Net Income': net_income, 'Depreciation': depreciation_val,
        'Operating CF': cfo, 'Investing CF': cfi, 'Financing CF': cff,
        'Net Change in Cash': net_cf, 'Ending Cash': cash,
    }, index=months_list).T

    return is_df, bs_df, cf_df, revenue, net_income, cash, ebit, ebit_margin, net_margin


# ============================================================================
# HISTORICAL DATA GENERATOR (for training forecast models)
# ============================================================================
def generate_historical_data(n_months=36, seed=42):
    """Generate 3 years of realistic historical revenue data with trend, seasonality, and noise."""
    np.random.seed(seed)
    t = np.arange(n_months)
    trend = 200_000 * (1.035 ** (t/12))                              # ~3.5% annual growth
    seasonal = 30_000 * np.sin(2 * np.pi * t / 12 - np.pi/2)          # yearly cycle
    q4_boost = np.where(t % 12 >= 9, 25_000, 0)                       # Q4 lift
    noise = np.random.normal(0, 12_000, n_months)
    shocks = np.zeros(n_months)
    shocks[15] = -40_000  # one-time shock
    shocks[28] = 35_000
    revenue = trend + seasonal + q4_boost + noise + shocks
    dates = pd.date_range(end=pd.Timestamp('2026-03-31'), periods=n_months, freq='ME')
    return pd.DataFrame({'ds': dates, 'y': revenue})


# ============================================================================
# FORECASTING MODELS
# ============================================================================
def forecast_prophet(hist_df, n_periods=12):
    """Facebook Prophet — handles trend, seasonality, holidays automatically."""
    if not FORECAST_AVAILABLE['prophet']:
        return None, "Prophet not installed"
    try:
        m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                    changepoint_prior_scale=0.05, interval_width=0.80)
        m.fit(hist_df)
        future = m.make_future_dataframe(periods=n_periods, freq='ME')
        fcst = m.predict(future)
        return fcst[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(n_periods).reset_index(drop=True), "OK"
    except Exception as e:
        return None, f"Prophet error: {e}"


def forecast_arima(hist_df, n_periods=12):
    """ARIMA(2,1,2) — classical Box-Jenkins with confidence intervals."""
    if not FORECAST_AVAILABLE['arima']:
        return None, "statsmodels not installed"
    try:
        model = ARIMA(hist_df['y'].values, order=(2, 1, 2),
                      seasonal_order=(1, 1, 1, 12) if len(hist_df) >= 24 else (0,0,0,0))
        fit = model.fit()
        fc = fit.get_forecast(steps=n_periods)
        mean = fc.predicted_mean
        ci = fc.conf_int(alpha=0.20)
        last_date = hist_df['ds'].max()
        future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=n_periods, freq='ME')
        return pd.DataFrame({
            'ds': future_dates, 'yhat': mean,
            'yhat_lower': ci[:, 0], 'yhat_upper': ci[:, 1]
        }), "OK"
    except Exception as e:
        return None, f"ARIMA error: {e}"


def forecast_lstm(hist_df, n_periods=12, lookback=6, epochs=50):
    """LSTM neural network — deep learning sequence model."""
    if not FORECAST_AVAILABLE['lstm']:
        return None, "TensorFlow not installed"
    try:
        tf.random.set_seed(42)
        np.random.seed(42)
        values = hist_df['y'].values.reshape(-1, 1)
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(values)
        X, y = [], []
        for i in range(lookback, len(scaled)):
            X.append(scaled[i-lookback:i, 0]); y.append(scaled[i, 0])
        X, y = np.array(X), np.array(y)
        X = X.reshape((X.shape[0], X.shape[1], 1))

        model = Sequential([
            LSTM(50, return_sequences=True, input_shape=(lookback, 1)),
            Dropout(0.2),
            LSTM(30, return_sequences=False),
            Dropout(0.2),
            Dense(20, activation='relu'),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        model.fit(X, y, epochs=epochs, batch_size=4, verbose=0)

        # Recursive multi-step forecast
        window = scaled[-lookback:].flatten().tolist()
        preds = []
        for _ in range(n_periods):
            xin = np.array(window[-lookback:]).reshape(1, lookback, 1)
            p = model.predict(xin, verbose=0)[0, 0]
            preds.append(p); window.append(p)
        preds = scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()

        # Empirical error band from training residuals
        train_preds = scaler.inverse_transform(model.predict(X, verbose=0))
        actual = scaler.inverse_transform(y.reshape(-1, 1))
        sigma = float(np.std(actual - train_preds))

        last_date = hist_df['ds'].max()
        future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=n_periods, freq='ME')
        return pd.DataFrame({
            'ds': future_dates, 'yhat': preds,
            'yhat_lower': preds - 1.28 * sigma,
            'yhat_upper': preds + 1.28 * sigma
        }), "OK"
    except Exception as e:
        return None, f"LSTM error: {e}"


def forecast_holtwinters(hist_df, n_periods=12):
    """Holt-Winters triple exponential smoothing — fast baseline."""
    if not FORECAST_AVAILABLE['arima']:
        return None, "statsmodels not installed"
    try:
        seasonal = 'add' if len(hist_df) >= 24 else None
        model = ExponentialSmoothing(hist_df['y'].values, trend='add',
                                      seasonal=seasonal, seasonal_periods=12 if seasonal else None)
        fit = model.fit()
        preds = fit.forecast(n_periods)
        resid_std = float(np.std(fit.resid))
        last_date = hist_df['ds'].max()
        future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=n_periods, freq='ME')
        return pd.DataFrame({
            'ds': future_dates, 'yhat': preds,
            'yhat_lower': preds - 1.28 * resid_std,
            'yhat_upper': preds + 1.28 * resid_std
        }), "OK"
    except Exception as e:
        return None, f"HW error: {e}"


def forecast_ensemble(forecasts_dict):
    """Weighted ensemble of all successful forecasts."""
    valid = {k: v for k, v in forecasts_dict.items() if v is not None}
    if not valid: return None
    first_key = list(valid.keys())[0]
    dates = valid[first_key]['ds']
    yhat = np.mean([v['yhat'].values for v in valid.values()], axis=0)
    lower = np.mean([v['yhat_lower'].values for v in valid.values()], axis=0)
    upper = np.mean([v['yhat_upper'].values for v in valid.values()], axis=0)
    return pd.DataFrame({'ds': dates, 'yhat': yhat, 'yhat_lower': lower, 'yhat_upper': upper})


def monte_carlo_simulation(base_forecast, n_sims=500, volatility=0.15):
    """Monte Carlo simulation around a point forecast."""
    if base_forecast is None: return None
    means = base_forecast['yhat'].values
    n = len(means)
    sims = np.zeros((n_sims, n))
    for i in range(n_sims):
        shocks = np.random.normal(1.0, volatility, n)
        sims[i] = means * np.cumprod(1 + (shocks - 1) * 0.3)  # dampened
    return {
        'sims': sims,
        'p10': np.percentile(sims, 10, axis=0),
        'p50': np.percentile(sims, 50, axis=0),
        'p90': np.percentile(sims, 90, axis=0),
        'dates': base_forecast['ds'].values
    }


def evaluate_backtest(hist_df, test_size=6):
    """Walk-forward validation on last N months — returns MAPE for each model."""
    if len(hist_df) < test_size + 12: return {}
    train = hist_df.iloc[:-test_size].reset_index(drop=True)
    test = hist_df.iloc[-test_size:].reset_index(drop=True)
    actual = test['y'].values
    results = {}

    def mape(a, p):
        return float(np.mean(np.abs((a - p) / a)) * 100)

    for name, fn in [('Prophet', forecast_prophet), ('ARIMA', forecast_arima),
                     ('Holt-Winters', forecast_holtwinters), ('LSTM', forecast_lstm)]:
        fc, status = fn(train, n_periods=test_size)
        if fc is not None:
            results[name] = {'mape': mape(actual, fc['yhat'].values), 'status': '✓'}
        else:
            results[name] = {'mape': None, 'status': '✗'}
    return results


# ============================================================================
# TABLE FORMATTING
# ============================================================================
def format_financial_df(df, highlight_rows=None, pct_rows=None):
    highlight_rows = highlight_rows or []
    pct_rows = pct_rows or []

    def fmt(v, is_pct=False):
        if is_pct: return f"{v:.1f}%"
        if abs(v) < 1: return f"{v:,.3f}"
        return f"{v:,.0f}"

    styled = df.copy()
    for idx in styled.index:
        is_pct = idx in pct_rows
        styled.loc[idx] = [fmt(v, is_pct) for v in df.loc[idx]]

    def row_style(row):
        n = row.name
        if n in highlight_rows:
            return ['background-color: #1e3a8a; color: white; font-weight: 700; padding: 8px;'] * len(row)
        if n in pct_rows:
            return ['color: #3b82f6; font-style: italic;'] * len(row)
        if n.startswith('Total') or n in ['Gross Profit', 'EBIT', 'EBT']:
            return ['background-color: #e0e7ff; font-weight: 600;'] * len(row)
        return [''] * len(row)

    return (styled.style.apply(row_style, axis=1)
            .set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', 'white'),
                                              ('font-weight', '600'), ('padding', '10px'), ('font-family', 'Segoe UI')]},
                {'selector': 'td', 'props': [('padding', '8px 12px'), ('text-align', 'right'),
                                              ('font-family', 'Segoe UI'), ('border-bottom', '1px solid #e2e8f0')]},
                {'selector': 'th.row_heading', 'props': [('text-align', 'left'), ('background-color', '#f8fafc'),
                                                         ('color', '#1e293b'), ('font-weight', '500')]},
            ]))


# ============================================================================
# WIDGETS
# ============================================================================
slider_layout = Layout(width='450px')
style_slider = {'description_width': '200px'}

preset = widgets.Dropdown(options=list(PRESETS.keys()) + ['Custom'], value='Base Case',
                           description='📋 Preset:', style={'description_width': '120px'},
                           layout=Layout(width='300px'))
view_mode = widgets.Dropdown(
    options=['All Statements', 'Income Statement', 'Balance Sheet', 'Cash Flow', 'Charts Only', 'Forecast Only'],
    value='All Statements', description='👁 View:',
    style={'description_width': '120px'}, layout=Layout(width='300px'))

rev_growth = widgets.FloatSlider(value=4.0, min=-5, max=20, step=0.5, description='Revenue Growth (%/mo)', style=style_slider, layout=slider_layout, readout_format='.1f')
cogs = widgets.FloatSlider(value=45.0, min=20, max=80, step=1, description='COGS (% of Revenue)', style=style_slider, layout=slider_layout, readout_format='.1f')
salaries = widgets.FloatSlider(value=15.0, min=5, max=40, step=1, description='Salaries (% of Revenue)', style=style_slider, layout=slider_layout, readout_format='.1f')
rent = widgets.IntSlider(value=50000, min=10000, max=200000, step=5000, description='Rent ($/month)', style=style_slider, layout=slider_layout)
marketing = widgets.FloatSlider(value=5.0, min=0, max=20, step=0.5, description='Marketing (% of Revenue)', style=style_slider, layout=slider_layout, readout_format='.1f')
depreciation = widgets.IntSlider(value=20000, min=0, max=100000, step=1000, description='Depreciation ($/month)', style=style_slider, layout=slider_layout)
other_opex = widgets.FloatSlider(value=3.0, min=0, max=15, step=0.5, description='Other OpEx (% of Revenue)', style=style_slider, layout=slider_layout, readout_format='.1f')
interest = widgets.IntSlider(value=10000, min=0, max=50000, step=1000, description='Interest Expense ($/mo)', style=style_slider, layout=slider_layout)
tax = widgets.FloatSlider(value=25.0, min=0, max=50, step=1, description='Tax Rate (%)', style=style_slider, layout=slider_layout, readout_format='.1f')
dso = widgets.IntSlider(value=35, min=0, max=120, step=1, description='Days Sales Outstanding', style=style_slider, layout=slider_layout)
dio = widgets.IntSlider(value=25, min=0, max=120, step=1, description='Days Inventory Outstanding', style=style_slider, layout=slider_layout)
dpo = widgets.IntSlider(value=30, min=0, max=120, step=1, description='Days Payable Outstanding', style=style_slider, layout=slider_layout)
capex = widgets.FloatSlider(value=3.0, min=0, max=20, step=0.5, description='CapEx (% of Revenue)', style=style_slider, layout=slider_layout, readout_format='.1f')
dividends = widgets.FloatSlider(value=10.0, min=0, max=100, step=5, description='Dividends (% of NI)', style=style_slider, layout=slider_layout, readout_format='.1f')

# Forecast controls
forecast_horizon = widgets.IntSlider(value=12, min=3, max=24, step=1, description='Forecast Horizon (mo)', style=style_slider, layout=slider_layout)
mc_volatility = widgets.FloatSlider(value=0.15, min=0.05, max=0.50, step=0.05, description='Monte Carlo Volatility', style=style_slider, layout=slider_layout, readout_format='.2f')
seasonality_toggle = widgets.Checkbox(value=True, description='Apply seasonality to model', style=style_slider)
run_forecast_btn = widgets.Button(description='🚀 Run Advanced Forecast', button_style='primary',
                                    layout=Layout(width='300px', height='40px'))
forecast_status = widgets.HTML(value='<i>Click "Run Advanced Forecast" to train models. This takes ~20-40 seconds.</i>')
forecast_output = widgets.Output()

# Preset handler
def apply_preset(change):
    if change['new'] != 'Custom' and change['new'] in PRESETS:
        v = PRESETS[change['new']]
        for k, w in [('rev_growth', rev_growth), ('cogs', cogs), ('salaries', salaries), ('rent', rent),
                     ('marketing', marketing), ('depreciation', depreciation), ('other_opex', other_opex),
                     ('interest', interest), ('tax', tax), ('dso', dso), ('dio', dio), ('dpo', dpo),
                     ('capex', capex), ('dividends', dividends)]:
            w.value = v[k]
preset.observe(apply_preset, names='value')


# ============================================================================
# MAIN RENDER FUNCTION
# ============================================================================
def render(rev_growth, cogs, salaries, rent, marketing, depreciation, other_opex,
           interest, tax, dso, dio, dpo, capex, dividends, view_mode, seasonality_toggle):

    is_df, bs_df, cf_df, revenue, net_income, cash, ebit, ebit_margin, net_margin = build_model(
        rev_growth, cogs, salaries, rent, marketing, depreciation, other_opex,
        interest, tax, dso, dio, dpo, capex, dividends, seasonality=seasonality_toggle
    )

    # KPI cards
    total_rev = sum(revenue); total_ni = sum(net_income)
    ending_cash = cash[-1]; avg_margin = np.mean(net_margin)
    kpi_html = f"""
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin: 20px 0;">
      <div style="background: linear-gradient(135deg, #3b82f6, #1e40af); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
        <div style="font-size: 12px; opacity: 0.9;">TOTAL REVENUE (12M)</div>
        <div style="font-size: 26px; font-weight: 700; margin-top: 8px;">${total_rev:,.0f}</div></div>
      <div style="background: linear-gradient(135deg, #10b981, #047857); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
        <div style="font-size: 12px; opacity: 0.9;">NET INCOME (12M)</div>
        <div style="font-size: 26px; font-weight: 700; margin-top: 8px;">${total_ni:,.0f}</div></div>
      <div style="background: linear-gradient(135deg, #8b5cf6, #6d28d9); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
        <div style="font-size: 12px; opacity: 0.9;">ENDING CASH</div>
        <div style="font-size: 26px; font-weight: 700; margin-top: 8px;">${ending_cash:,.0f}</div></div>
      <div style="background: linear-gradient(135deg, #f59e0b, #d97706); color: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1);">
        <div style="font-size: 12px; opacity: 0.9;">AVG NET MARGIN</div>
        <div style="font-size: 26px; font-weight: 700; margin-top: 8px;">{avg_margin:.1f}%</div></div>
    </div>
    """
    display(HTML(kpi_html))

    # Charts
    if view_mode in ['All Statements', 'Charts Only']:
        months_list = [f"M{i+1}" for i in range(12)]
        fig = make_subplots(rows=2, cols=2,
            subplot_titles=('Revenue & Net Income', 'Margin Trends', 'Cash Balance', 'EBIT over Time'))
        fig.add_trace(go.Bar(x=months_list, y=revenue, name='Revenue', marker_color='#3b82f6'), 1, 1)
        fig.add_trace(go.Scatter(x=months_list, y=net_income, name='Net Income', mode='lines+markers',
                                  line=dict(color='#10b981', width=3)), 1, 1)
        fig.add_trace(go.Scatter(x=months_list, y=ebit_margin, name='EBIT Margin %', mode='lines+markers',
                                  line=dict(color='#8b5cf6', width=3)), 1, 2)
        fig.add_trace(go.Scatter(x=months_list, y=net_margin, name='Net Margin %', mode='lines+markers',
                                  line=dict(color='#f59e0b', width=3)), 1, 2)
        fig.add_trace(go.Scatter(x=months_list, y=cash, name='Cash', mode='lines+markers',
                                  fill='tozeroy', line=dict(color='#06b6d4', width=3)), 2, 1)
        fig.add_trace(go.Bar(x=months_list, y=ebit, name='EBIT',
                             marker_color=['#10b981' if v >= 0 else '#ef4444' for v in ebit]), 2, 2)
        fig.update_layout(height=600, showlegend=True, template='plotly_white',
                           title_text="<b>Financial Performance Dashboard</b>",
                           title_font_size=18, font=dict(family='Segoe UI'))
        fig.show()

    if view_mode in ['All Statements', 'Income Statement']:
        display(HTML('<div class="section-header">📊 Income Statement</div>'))
        display(format_financial_df(is_df, highlight_rows=['Net Income'],
                                     pct_rows=['Gross Margin %', 'EBIT Margin %', 'Net Margin %']))
    if view_mode in ['All Statements', 'Balance Sheet']:
        display(HTML('<div class="section-header">🏦 Balance Sheet</div>'))
        display(format_financial_df(bs_df, highlight_rows=['Total Assets', 'Total Equity']))
    if view_mode in ['All Statements', 'Cash Flow']:
        display(HTML('<div class="section-header">💰 Cash Flow Statement</div>'))
        display(format_financial_df(cf_df, highlight_rows=['Ending Cash']))


# ============================================================================
# FORECAST RUNNER (on-demand because it's slow)
# ============================================================================
def run_forecasts(btn):
    with forecast_output:
        clear_output()
        forecast_status.value = '<span class="status-good">⏳ Training models on 36 months of historical data...</span>'

        hist = generate_historical_data(n_months=36)
        horizon = forecast_horizon.value

        # Show historical data
        display(HTML('<div class="ml-header">📜 Historical Revenue (36 months of training data)</div>'))
        fig0 = go.Figure()
        fig0.add_trace(go.Scatter(x=hist['ds'], y=hist['y'], mode='lines+markers',
                                   line=dict(color='#1e40af', width=2), marker=dict(size=6),
                                   name='Historical Revenue'))
        fig0.update_layout(template='plotly_white', height=300,
                            title='Synthetic Historical Revenue Data',
                            yaxis_tickformat='$,.0f', font=dict(family='Segoe UI'))
        fig0.show()

        # Run all models
        models_status = []
        forecast_status.value = '<span class="status-good">⏳ Running Prophet...</span>'
        prophet_fc, p_stat = forecast_prophet(hist, horizon)
        models_status.append(('Prophet', p_stat))

        forecast_status.value = '<span class="status-good">⏳ Running ARIMA...</span>'
        arima_fc, a_stat = forecast_arima(hist, horizon)
        models_status.append(('ARIMA', a_stat))

        forecast_status.value = '<span class="status-good">⏳ Running Holt-Winters...</span>'
        hw_fc, hw_stat = forecast_holtwinters(hist, horizon)
        models_status.append(('Holt-Winters', hw_stat))

        forecast_status.value = '<span class="status-good">⏳ Training LSTM neural network (50 epochs)...</span>'
        lstm_fc, l_stat = forecast_lstm(hist, horizon)
        models_status.append(('LSTM', l_stat))

        forecast_status.value = '<span class="status-good">⏳ Building ensemble & Monte Carlo simulation...</span>'
        ensemble_fc = forecast_ensemble({'prophet': prophet_fc, 'arima': arima_fc,
                                          'hw': hw_fc, 'lstm': lstm_fc})
        mc = monte_carlo_simulation(ensemble_fc, n_sims=500, volatility=mc_volatility.value)

        # Status table
        status_html = '<div class="ml-header">🤖 Model Status</div><table style="width:100%; font-family: Segoe UI; border-collapse: collapse;">'
        status_html += '<tr style="background:#1e293b;color:white;"><th style="padding:10px;text-align:left;">Model</th><th style="padding:10px;text-align:left;">Status</th><th style="padding:10px;text-align:left;">Description</th></tr>'
        descriptions = {
            'Prophet': 'Decomposable model: trend + seasonality + holidays (Meta/Facebook)',
            'ARIMA': 'Classical statistical model with autoregression & moving averages',
            'Holt-Winters': 'Triple exponential smoothing — fast baseline',
            'LSTM': 'Deep learning recurrent neural network with dropout regularization'
        }
        for name, stat in models_status:
            ok = stat == 'OK'
            badge = f'<span class="status-good">✓ Trained</span>' if ok else f'<span class="status-bad">✗ {stat}</span>'
            status_html += f'<tr style="border-bottom:1px solid #e2e8f0;"><td style="padding:10px;font-weight:600;">{name}</td><td style="padding:10px;">{badge}</td><td style="padding:10px;color:#64748b;">{descriptions[name]}</td></tr>'
        status_html += '</table>'
        display(HTML(status_html))

        # Combined forecast chart
        display(HTML('<div class="ml-header">🔮 Revenue Forecast — All Models Compared</div>'))
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(x=hist['ds'], y=hist['y'], mode='lines+markers',
                                   line=dict(color='#1e293b', width=2), name='Historical',
                                   marker=dict(size=5)))
        colors = {'Prophet': '#3b82f6', 'ARIMA': '#ef4444',
                  'Holt-Winters': '#f59e0b', 'LSTM': '#8b5cf6', 'Ensemble': '#10b981'}
        for name, fc in [('Prophet', prophet_fc), ('ARIMA', arima_fc),
                         ('Holt-Winters', hw_fc), ('LSTM', lstm_fc), ('Ensemble', ensemble_fc)]:
            if fc is not None:
                w = 4 if name == 'Ensemble' else 2
                dash = 'solid' if name == 'Ensemble' else 'dash'
                fig1.add_trace(go.Scatter(x=fc['ds'], y=fc['yhat'], mode='lines+markers',
                                           line=dict(color=colors[name], width=w, dash=dash), name=name))
        # Ensemble confidence band
        if ensemble_fc is not None:
            fig1.add_trace(go.Scatter(x=list(ensemble_fc['ds']) + list(ensemble_fc['ds'][::-1]),
                                       y=list(ensemble_fc['yhat_upper']) + list(ensemble_fc['yhat_lower'][::-1]),
                                       fill='toself', fillcolor='rgba(16,185,129,0.15)',
                                       line=dict(color='rgba(255,255,255,0)'), name='Ensemble 80% CI',
                                       showlegend=True, hoverinfo='skip'))
        fig1.update_layout(template='plotly_white', height=500,
                            title=f'<b>{horizon}-Month Revenue Forecast — Multi-Model Comparison</b>',
                            yaxis_tickformat='$,.0f', hovermode='x unified',
                            font=dict(family='Segoe UI'))
        fig1.show()

        # Monte Carlo fan chart
        if mc is not None:
            display(HTML('<div class="ml-header">🎲 Monte Carlo Simulation (500 paths)</div>'))
            fig2 = go.Figure()
            # Sample paths
            for i in range(0, 500, 10):
                fig2.add_trace(go.Scatter(x=mc['dates'], y=mc['sims'][i], mode='lines',
                                           line=dict(color='rgba(139,92,246,0.06)', width=1),
                                           showlegend=False, hoverinfo='skip'))
            fig2.add_trace(go.Scatter(x=mc['dates'], y=mc['p90'], mode='lines',
                                       line=dict(color='#8b5cf6', width=2, dash='dash'), name='P90'))
            fig2.add_trace(go.Scatter(x=mc['dates'], y=mc['p50'], mode='lines',
                                       line=dict(color='#1e40af', width=3), name='P50 (Median)'))
            fig2.add_trace(go.Scatter(x=mc['dates'], y=mc['p10'], mode='lines',
                                       line=dict(color='#ef4444', width=2, dash='dash'), name='P10'))
            fig2.update_layout(template='plotly_white', height=450,
                                title=f'<b>Monte Carlo Revenue Projection (volatility σ={mc_volatility.value:.2f})</b>',
                                yaxis_tickformat='$,.0f', font=dict(family='Segoe UI'))
            fig2.show()

        # Backtest
        display(HTML('<div class="ml-header">📏 Backtest Accuracy (last 6 months held out)</div>'))
        forecast_status.value = '<span class="status-good">⏳ Running walk-forward backtest...</span>'
        bt = evaluate_backtest(hist, test_size=6)
        if bt:
            rows = ''
            sorted_bt = sorted(bt.items(), key=lambda x: (x[1]['mape'] is None, x[1]['mape'] or 999))
            for i, (name, r) in enumerate(sorted_bt):
                mape = f"{r['mape']:.2f}%" if r['mape'] is not None else "—"
                medal = '🥇' if i == 0 and r['mape'] is not None else ('🥈' if i == 1 and r['mape'] is not None else ('🥉' if i == 2 and r['mape'] is not None else ''))
                bg = '#dcfce7' if i == 0 and r['mape'] is not None else 'white'
                rows += f'<tr style="background:{bg};border-bottom:1px solid #e2e8f0;"><td style="padding:10px;font-weight:600;">{medal} {name}</td><td style="padding:10px;text-align:right;font-family:monospace;font-size:15px;">{mape}</td><td style="padding:10px;">{r["status"]}</td></tr>'
            bt_html = f'<table style="width:100%;font-family:Segoe UI;border-collapse:collapse;"><tr style="background:#1e293b;color:white;"><th style="padding:10px;text-align:left;">Model</th><th style="padding:10px;text-align:right;">MAPE (lower is better)</th><th style="padding:10px;text-align:left;">Status</th></tr>{rows}</table>'
            display(HTML(bt_html))

        # Forecast summary table
        if ensemble_fc is not None:
            display(HTML('<div class="ml-header">📋 Ensemble Forecast Values</div>'))
            summary = pd.DataFrame({
                'Month': [d.strftime('%Y-%m') for d in ensemble_fc['ds']],
                'Forecast': [f"${v:,.0f}" for v in ensemble_fc['yhat']],
                'Lower (P10)': [f"${v:,.0f}" for v in ensemble_fc['yhat_lower']],
                'Upper (P90)': [f"${v:,.0f}" for v in ensemble_fc['yhat_upper']],
            })
            display(summary.style.set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#1e293b'), ('color', 'white'),
                                              ('padding', '10px'), ('font-family', 'Segoe UI')]},
                {'selector': 'td', 'props': [('padding', '8px 12px'), ('text-align', 'right'),
                                              ('font-family', 'Segoe UI')]}
            ]).hide(axis='index'))

        forecast_status.value = '<span class="status-good">✅ All models trained. Rerun with different settings anytime.</span>'

run_forecast_btn.on_click(run_forecasts)


# ============================================================================
# LAYOUT
# ============================================================================
display(HTML(f"""
<div class="model-header">
  <div style="font-size: 30px; font-weight: 700;">📊 Financial Model + ML Forecasting</div>
  <div style="font-size: 14px; opacity: 0.95; margin-top: 6px;">
    3-statement model · Interactive scenarios · Prophet · LSTM · ARIMA · Holt-Winters · Ensemble · Monte Carlo
  </div>
  <div style="font-size: 12px; opacity: 0.85; margin-top: 8px;">
    Forecasting libraries — Prophet: {'✅' if FORECAST_AVAILABLE['prophet'] else '❌ (run pip install)'} ·
    statsmodels (ARIMA): {'✅' if FORECAST_AVAILABLE['arima'] else '❌'} ·
    TensorFlow (LSTM): {'✅' if FORECAST_AVAILABLE['lstm'] else '❌'}
  </div>
</div>
"""))

display(HTML('<div class="section-header">⚙️ Scenario & View Controls</div>'))
display(HBox([preset, view_mode, seasonality_toggle]))

display(HTML('<div class="section-header">📈 Income Statement Assumptions</div>'))
display(HBox([VBox([rev_growth, cogs, salaries, rent, marketing]),
              VBox([depreciation, other_opex, interest, tax])]))

display(HTML('<div class="section-header">🏦 Balance Sheet & Cash Flow Assumptions</div>'))
display(HBox([VBox([dso, dio, dpo]), VBox([capex, dividends])]))

out = interactive_output(render, {
    'rev_growth': rev_growth, 'cogs': cogs, 'salaries': salaries, 'rent': rent,
    'marketing': marketing, 'depreciation': depreciation, 'other_opex': other_opex,
    'interest': interest, 'tax': tax, 'dso': dso, 'dio': dio, 'dpo': dpo,
    'capex': capex, 'dividends': dividends, 'view_mode': view_mode,
    'seasonality_toggle': seasonality_toggle
})
display(out)

display(HTML('<div class="ml-header">🤖 Advanced ML Forecasting Controls</div>'))
display(HBox([VBox([forecast_horizon, mc_volatility]),
              VBox([run_forecast_btn, forecast_status])]))
display(forecast_output)

Output()

Output()